### Reading Datasets

In [2]:
import os

os.makedirs(os.path.join("..",'data'), exist_ok=True)
data_file = os.path.join('..','data','house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('''NumRooms,RoofType,Price
NA,NA,127500
2,NA,106000
4,Slate,178100
NA,NA,140000''')
    
import pandas as pd
data = pd.read_csv(data_file)
print(data)

   NumRooms RoofType   Price
0       NaN      NaN  127500
1       2.0      NaN  106000
2       4.0    Slate  178100
3       NaN      NaN  140000


In [3]:
inputs, targets = data.iloc[:,0:2], data.iloc[:,2]
inputs = pd.get_dummies(inputs,dummy_na=True) 
print(inputs)

   NumRooms  RoofType_Slate  RoofType_nan
0       NaN           False          True
1       2.0           False          True
2       4.0            True         False
3       NaN           False          True


In [4]:
inputs = inputs.fillna(inputs.mean())
print(inputs)

   NumRooms  RoofType_Slate  RoofType_nan
0       3.0           False          True
1       2.0           False          True
2       4.0            True         False
3       3.0           False          True


In [5]:
import torch

X=torch.tensor(inputs.to_numpy(dtype=float))
y=torch.tensor(targets.to_numpy(dtype=float))
X,y

(tensor([[3., 0., 1.],
         [2., 0., 1.],
         [4., 1., 0.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

### Exercises

1. Try loading datasets, e.g., Abalone from the UCI Machine Learning Repository, and inspect their properties. What fraction of them has missing values? What fraction of the variables is numerical, categorical, or text?

In [1]:
%pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\Py Torch\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [32]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
abalone = fetch_ucirepo(id=1) 
data = abalone.data
data

{'ids': None,
 'features':      Sex  Length  Diameter  Height  Whole_weight  Shucked_weight  \
 0      M   0.455     0.365   0.095        0.5140          0.2245   
 1      M   0.350     0.265   0.090        0.2255          0.0995   
 2      F   0.530     0.420   0.135        0.6770          0.2565   
 3      M   0.440     0.365   0.125        0.5160          0.2155   
 4      I   0.330     0.255   0.080        0.2050          0.0895   
 ...   ..     ...       ...     ...           ...             ...   
 4172   F   0.565     0.450   0.165        0.8870          0.3700   
 4173   M   0.590     0.440   0.135        0.9660          0.4390   
 4174   M   0.600     0.475   0.205        1.1760          0.5255   
 4175   F   0.625     0.485   0.150        1.0945          0.5310   
 4176   M   0.710     0.555   0.195        1.9485          0.9455   
 
       Viscera_weight  Shell_weight  
 0             0.1010        0.1500  
 1             0.0485        0.0700  
 2             0.1415        0

In [33]:
import pandas as pd
from pandas.api.types import is_numeric_dtype, is_string_dtype

if not isinstance(data, (pd.DataFrame)):
    data = pd.DataFrame(data.features)
    
num_missing, total = 0, data.shape[1]
numerical, string, categorical = 0, 0, 0

for property in data:
    data_prop = data[property]
    if data_prop.isna().sum() > 0:
        num_missing += 1
    if is_numeric_dtype(data_prop): numerical += 1
    elif is_string_dtype(data_prop): string += 1
    else: categorical += 1
    
print(f"{(num_missing / total * 100):.2f}% of properties have missing data.")
print(f"{(numerical / total * 100):.2f} of properties are numerical, {(categorical / total * 100):.2f} are categorical, {(string / total * 100):.2f} are string.")

0.00% of properties have missing data.
87.50 of properties are numerical, 0.00 are categorical, 12.50 are string.


2. Try indexing and selecting data columns by name rather than by column number. The pandas documentation on indexing has further details on how to do this.

In [ ]:
# Index number -> data.iloc[:,i]
# Name -> data[name]

3. How large a dataset do you think you could load this way? What might be the limitations? Hint: consider time to read the data, respresentation, processing, and memory footprint. Try this out on your laptop. What happens if you try it out on a server?

I believe datasets that are a few gigabytes in size or larger could be loaded this way. Limitations are user network bandwidth and laptop storage. 

On a server, the network bandwidth and hardware capabilities would be much higher. Thus, datasets could be hundreds of gigabytes in size.

4. How would you deal with data that has a very large number of categories? What if the category labels are all unique? Should you include the latter?

If the data has a very large number of categories, I would perform a singular value decomposition (SVD) and construct a scree plot to determine which features are the most important as well as how much each category contributes to the overall variance. Categories that contribute very little to the variance may be discarded to improve performance of machine learning algorithms. If all category labels are unique, then each category label has no correlation with any of the other features and thus can be discarded. An example of this is ID, which is often discarded due to having no relevance to any other variables. We should not include these categories of purely unique variables.

5. What alternatives to pandas can you think of? How about loading NumPy tensors from a file? Check out Piullow, the Python Imaging Library.

1. Polars
2. Numpy ndarrays
3. Higher order torch tensors